In [9]:
# ITR calculation function
import numpy as np

def itr(n, p, t):
    """
    Calculate information transfer rate (ITR) for BCI.
    n   : # of targets
    p   : target identification accuracy (0 <= p <= 1)
    t   : averaged time for a selection [s]
    returns itr [bits/min]
    """
    if p < 0 or p > 1:
        raise ValueError("Accuracy must be between 0 and 1.")
    elif p < 1/n:
        print("Warning: accuracy < chance level. ITR may be incorrect.")
        return 0
    elif p == 1:
        return np.log2(n)*60/t
    else:
        return (np.log2(n) + p*np.log2(p) + (1-p)*np.log2((1-p)/(n-1)))*60/t

In [10]:
# Filter bank decomposition
from scipy.signal import cheby1, cheb1ord, filtfilt

def filterbank(eeg, fs, idx_fb=1):
    """
    eeg: np.array [channels x samples] or [channels x samples x trials]
    fs: sampling rate
    idx_fb: filter bank index (1-10)
    """
    if idx_fb < 1 or idx_fb > 10:
        raise ValueError("Filter index must be 1 <= idx_fb <= 10")
    
    passband = np.array([6, 14, 22, 30, 38, 46, 54, 62, 70, 78])
    stopband = np.array([4, 10, 16, 24, 32, 40, 48, 56, 64, 72])
    
    fs2 = fs / 2
    Wp = [passband[idx_fb-1]/fs2, 90/fs2]
    Ws = [stopband[idx_fb-1]/fs2, 100/fs2]
    N, Wn = cheb1ord(Wp, Ws, 3, 40)
    B, A = cheby1(N, 0.5, Wn, btype='bandpass')
    
    if eeg.ndim == 2:
        y = np.array([filtfilt(B, A, ch) for ch in eeg])
    elif eeg.ndim == 3:
        y = np.zeros_like(eeg)
        for trial_i in range(eeg.shape[2]):
            for ch_i in range(eeg.shape[0]):
                y[ch_i,:,trial_i] = filtfilt(B, A, eeg[ch_i,:,trial_i])
    else:
        raise ValueError("EEG array must be 2D or 3D")
    return y

In [11]:
# FBCCA single trial score
from sklearn.cross_decomposition import CCA

def fbcca_score_single_trial(testdata, list_freqs, fs, num_harms, num_fbs):
    """
    testdata: [channels x timepoints]
    list_freqs: list of target frequencies
    """
    fb_coefs = np.array([fb_i**(-1.25)+0.25 for fb_i in range(1, num_fbs+1)])
    num_targs = len(list_freqs)
    num_smpls = testdata.shape[1]
    t = np.arange(num_smpls) / fs
    rho = np.zeros((num_fbs, num_targs))
    
    for targ_i, freq in enumerate(list_freqs):
        refdata = []
        for harm_i in range(1, num_harms+1):
            refdata.append(np.sin(2*np.pi*harm_i*freq*t))
            refdata.append(np.cos(2*np.pi*harm_i*freq*t))
        refdata = np.array(refdata).T  # timepoints x 2*num_harms
        
        for fb_i in range(num_fbs):
            testdata_fb = filterbank(testdata, fs, fb_i+1)
            std_ch = np.std(testdata_fb, axis=1)
            good_ch = std_ch > 1e-8
            testdata_fb = testdata_fb[good_ch,:]
            if testdata_fb.shape[0] < 1:
                r_tmp = 0
            else:
                cca = CCA(n_components=1)
                cca.fit(testdata_fb.T, refdata)
                U, V = cca.transform(testdata_fb.T, refdata)
                r_tmp = np.corrcoef(U.T, V.T)[0,1]
            rho[fb_i, targ_i] = r_tmp
    
    score = np.sum(fb_coefs * (rho[:,0]**2))
    return score

# main

In [29]:
import numpy as np
import scipy
import scipy.io
import numpy as np

txt = 'four_85_ssvep'
eeg_trials = np.load(f'data/eeg_trials_{txt}.npy', allow_pickle=True)
eeg_labels = np.load(f'data/eeg_labels_{txt}.npy')
print(eeg_trials.shape)

(107, 8, 497)


In [30]:
list_freqs = [15, 12, 10, 8.57]
num_fbs = 5
num_harms = 4
fs = 250
len_gaze_s = 1.5
len_shift_s = 0.5
len_sel_s = len_gaze_s + len_shift_s

num_trials = len(eeg_labels)
num_freqs = len(list_freqs)

In [31]:
estimated = []

for trial_i in range(num_trials):    
    trial_EEG = eeg_trials[trial_i]
    start = int(len_shift_s * fs)
    end = int(len_sel_s * fs)
    trial_EEG = trial_EEG[:, start:end]
    estimated_targets = [fbcca_score_single_trial(trial_EEG, [list_freqs[freq_idx]], fs, num_harms, num_fbs) for freq_idx in range(num_freqs)]
    idx_max = np.argmax(estimated_targets)
    estimated.append(idx_max)
    
    print(f"predict: {idx_max} \t ture: {eeg_labels[trial_i]}")
print(estimated[0])

predict: 1 	 ture: 0
predict: 1 	 ture: 2
predict: 3 	 ture: 2
predict: 2 	 ture: 3
predict: 2 	 ture: 2
predict: 1 	 ture: 1
predict: 0 	 ture: 2
predict: 1 	 ture: 1
predict: 1 	 ture: 1
predict: 3 	 ture: 3
predict: 3 	 ture: 3
predict: 2 	 ture: 2
predict: 1 	 ture: 1
predict: 0 	 ture: 3
predict: 2 	 ture: 2
predict: 2 	 ture: 0
predict: 3 	 ture: 1
predict: 3 	 ture: 3
predict: 0 	 ture: 0
predict: 0 	 ture: 0
predict: 2 	 ture: 2
predict: 0 	 ture: 0
predict: 1 	 ture: 1
predict: 2 	 ture: 2
predict: 2 	 ture: 2
predict: 0 	 ture: 0
predict: 3 	 ture: 0
predict: 3 	 ture: 3
predict: 1 	 ture: 2
predict: 2 	 ture: 3
predict: 3 	 ture: 0
predict: 1 	 ture: 1
predict: 3 	 ture: 3
predict: 1 	 ture: 1
predict: 3 	 ture: 2
predict: 2 	 ture: 2
predict: 1 	 ture: 2
predict: 1 	 ture: 1
predict: 1 	 ture: 1
predict: 0 	 ture: 0
predict: 0 	 ture: 0
predict: 2 	 ture: 2
predict: 2 	 ture: 2
predict: 0 	 ture: 3
predict: 1 	 ture: 2
predict: 1 	 ture: 1
predict: 2 	 ture: 2
predict: 1 	 

In [33]:
correct = np.sum(estimated == eeg_labels)
print(correct)  # 2
print(len(eeg_labels))
print(correct/len(eeg_labels))


72
107
0.6728971962616822


In [ ]:
# # Main FBCCA evaluation script
# import numpy as np
# import scipy
# import scipy.io

# # load your EEG data here
# filename = r'D:\documents\vibe\analyse\eeg_data_four_61_ssvep.mat'
# mat = scipy.io.loadmat(filename)
# eeg = mat['eeg']
# list_freqs = [15, 13, 11, 8.57]

# print('EEG shape:', eeg.shape)
# print('Frequencies:', list_freqs)

# num_fbs = 5
# num_harms = 4
# fs = 250
# len_gaze_s = 1.5
# len_shift_s = 0.5
# len_sel_s = len_gaze_s + len_shift_s

# num_blocks = eeg.shape[3]
# num_targs = len(list_freqs)
# labels = np.arange(1, num_targs+1)

# len_gaze_smpl = round(len_gaze_s*fs)
# len_delay_smpl = round(0.2*fs)
# segment_data = slice(len_delay_smpl, len_delay_smpl+len_gaze_smpl)
# eeg = eeg[:,:,segment_data,:]

# estimated_all = []
# accs = []
# itrs = []

# for block_i in range(num_blocks):
#     estimated = []
#     for freq in range(eeg.shape[0]):
#         trial_EEG = eeg[freq,:,:,block_i]
#         estimated_targets = [fbcca_score_single_trial(trial_EEG, [list_freqs[freq_idx]], fs, num_harms, num_fbs) for freq_idx in range(num_targs)]
#         idx_max = np.argmax(estimated_targets)
#         estimated.append(idx_max+1)
#     estimated_all.append(estimated)
#     is_correct = np.array(estimated) == labels
#     accs.append(is_correct.mean())
#     itrs.append(itr(num_targs, is_correct.mean(), len_sel_s))
#     print(f'Trial {block_i+1}: Accuracy = {is_correct.mean()*100:.2f}%, ITR = {itrs[-1]:.2f} bpm')

# print(f'Mean accuracy = {np.mean(accs)*100:.2f}%')
# print(f'Mean ITR = {np.mean(itrs):.2f} bpm')

In [ ]:
# import numpy as np
# import scipy.io

# # Load EEG data
# filename = r'D:\documents\vibe\analyse\eeg_data_four_610_ssvep.mat'
# mat = scipy.io.loadmat(filename)
# eeg = mat['eeg']  # shape: [blocks, freqs, channels, samples]
# list_freqs = [15, 11, 13, 9]
# # list_freqs = [15, 13, 11, 9, 10]

# num_freqs, num_chans, num_samples, num_blocks = eeg.shape
# # Flatten block*freq and create labels
# eeg_flat = eeg.transpose(3, 0, 1, 2) #(num_blocks, num_freqs, num_chans, num_samples)

# test_two = [0, 1, 2, 3]
# eeg_flat = eeg_flat[:, test_two, :, :]
# list_freqs = np.array([list_freqs[i] for i in test_two])

# row = np.array(list(range(0, len(test_two))))
# num_blocks = eeg_flat.shape[0]
# labels = np.tile(row, (num_blocks, 1))
# labels = labels.reshape(-1)
# eeg_flat = eeg_flat.reshape(-1, num_chans, num_samples)

# # Random shuffle
# perm = np.random.permutation(len(labels))
# eeg_flat = eeg_flat[perm]
# labels = labels[perm]

# print('EEG shape after flattening:', eeg_flat.shape)
# print('Labels shape:', labels.shape)
# print(labels[:10])  # Print first 10 labels to verify
# # Parameters
# num_fbs = 5
# num_harms = 4
# fs = 250
# len_gaze_s = 1.5
# len_shift_s = 0.5
# len_sel_s = len_gaze_s + len_shift_s
# print('EEG shape after flattening:', eeg_flat.shape)
# print('Labels shape:', labels.shape)
# # Segment EEG (keep enough length to avoid filtfilt padlen issues)
# len_gaze_smpl = round(len_gaze_s*fs)
# len_delay_smpl = round(0.2*fs)
# segment_data = slice(len_delay_smpl, len_delay_smpl+len_gaze_smpl)
# eeg_flat = eeg_flat[:, :, segment_data]

# # FBCCA single-trial test
# estimated_all = []
# accs = []
# itrs = []

# for trial_i in range(eeg_flat.shape[0]):
#     trial_EEG = eeg_flat[trial_i]
#     estimated_targets = [fbcca_score_single_trial(trial_EEG, [f], fs, num_harms, num_fbs) for f in list_freqs]
#     idx_max = np.argmax(estimated_targets)
#     estimated_label = idx_max
#     estimated_all.append(estimated_label)
    
#     is_correct = (estimated_label == labels[trial_i])
#     accs.append(is_correct)
#     itrs.append(itr(len(list_freqs), float(is_correct), len_sel_s))

#     print(f'Trial {trial_i+1}: True = {labels[trial_i]}, Pred = {estimated_label}, Acc = {is_correct}, ITR = {itrs[-1]:.2f} bpm')

# print(f'Mean accuracy = {np.mean(accs)*100:.2f}%')
# print(f'Mean ITR = {np.mean(itrs):.2f} bpm')


In [ ]:
list_freqs

[15, 12, 10, 8.57]